# Catan Player Profiling
This notebook aggregates the game summaries and creates a dataframe profiling each player with various statistics.

In [20]:
import polars as pl
import glob
import os

# 1. Aggregate all the results across different games
out_dir = '../../data/game_log_dfs'
parquet_files = glob.glob(os.path.join(out_dir, '*.parquet'))

dfs = []
for p in parquet_files:
    # If the file hasn't been re-generated yet, it might not exist if they ran csv logic instead
    # However we added the parquet generator previously
    # Extract game id from "game_{game_id}_stats.parquet"
    game_id = os.path.basename(p).split('_')[1]
    try:
        df_game = pl.read_parquet(p)
        df_game = df_game.with_columns(pl.lit(game_id).alias("game_id"))
        dfs.append(df_game)
    except Exception as e:
        print(f"Error loading {p}: {e}")

if dfs:
    df_all = pl.concat(dfs, how="diagonal")
    print(f"Aggregated {len(dfs)} games.")


Aggregated 28 games.


In [21]:
if not df_all.is_empty():
    # Helper to calculate metrics per player
    
    # First calculate the flat player aggs
    player_profiles = df_all.group_by('player').agg(
        num_games=pl.len(),
        avg_turns_before_first_trade=pl.col('turns_before_first_trade').filter(pl.col('turns_before_first_trade') != -1).mean(),
        ratio_cards_given_to_taken=(pl.col('cards_lost_in_trades').sum() / pl.col('cards_gained_in_trades').sum()),
        trade_success_rate=(pl.col('trades_completed').sum() / pl.col('trades_proposed').sum()),
        avg_counter_offers=pl.col('counter_offers').mean(),
        avg_bank_trades=pl.col('bank_trades').mean(),
        avg_dev_cards_bought=pl.col('dev_cards_bought').mean(),
        avg_roads_built=pl.col('roads_built').mean(),
        avg_cities_built=pl.col('cities_built').mean(),
        avg_players_targeted=pl.col('unique_players_stolen_from').mean(),
        avg_times_targeted=pl.col('times_targeted_by_others').mean(),
        avg_cards_discarded_to_7=pl.col('resources_lost_to_7_roll').mean(),
        win_rate_largest_army=(pl.col('largest_army_received') > 0).sum() / pl.len(),
        win_rate_longest_road=(pl.col('longest_road_received') > 0).sum() / pl.len(),
        
        # NEW: Hand Size Trend
        overall_avg_hand_size=pl.col('avg_hand_size').mean(),
        
        # We need these to calculate top 3 starting resources per player
        total_starting_ore=pl.col('starting_ore').fill_null(0).sum(),
        total_starting_wool=pl.col('starting_wool').fill_null(0).sum(),
        total_starting_lumber=pl.col('starting_lumber').fill_null(0).sum(),
        total_starting_grain=pl.col('starting_grain').fill_null(0).sum(),
        total_starting_brick=pl.col('starting_brick').fill_null(0).sum(),
        
        # NEW: Resource Scarcity additions for trades
        total_traded_away_ore=pl.col('traded_away_ore').fill_null(0).sum(),
        total_traded_away_wool=pl.col('traded_away_wool').fill_null(0).sum(),
        total_traded_away_lumber=pl.col('traded_away_lumber').fill_null(0).sum(),
        total_traded_away_grain=pl.col('traded_away_grain').fill_null(0).sum(),
        total_traded_away_brick=pl.col('traded_away_brick').fill_null(0).sum(),
        
        total_received_trade_ore=pl.col('received_trade_ore').fill_null(0).sum(),
        total_received_trade_wool=pl.col('received_trade_wool').fill_null(0).sum(),
        total_received_trade_lumber=pl.col('received_trade_lumber').fill_null(0).sum(),
        total_received_trade_grain=pl.col('received_trade_grain').fill_null(0).sum(),
        total_received_trade_brick=pl.col('received_trade_brick').fill_null(0).sum()
    )
    
    # Calculate top 3 starting resources and trading behaviors
    derived_metrics_list = []
    
    for row in player_profiles.iter_rows(named=True):
        player = row['player']
        
        # Starting resources
        res_counts = {
            'ore': row['total_starting_ore'],
            'wool': row['total_starting_wool'],
            'lumber': row['total_starting_lumber'],
            'grain': row['total_starting_grain'],
            'brick': row['total_starting_brick']
        }
        sorted_res = sorted(res_counts.items(), key=lambda item: item[1], reverse=True)
        top_3 = [res for res, count in sorted_res[:3]]
        
        # Traded away stats
        traded_away = {
            'ore': row['total_traded_away_ore'],
            'wool': row['total_traded_away_wool'],
            'lumber': row['total_traded_away_lumber'],
            'grain': row['total_traded_away_grain'],
            'brick': row['total_traded_away_brick']
        }
        most_traded_away = max(traded_away, key=traded_away.get) if sum(traded_away.values()) > 0 else "None"
        
        # Received stats
        received_trade = {
            'ore': row['total_received_trade_ore'],
            'wool': row['total_received_trade_wool'],
            'lumber': row['total_received_trade_lumber'],
            'grain': row['total_received_trade_grain'],
            'brick': row['total_received_trade_brick']
        }
        most_received = max(received_trade, key=received_trade.get) if sum(received_trade.values()) > 0 else "None"
        
        derived_metrics_list.append({
            'player': player,
            'top_3_starting_resources': ", ".join(top_3),
            'most_traded_away_resource': most_traded_away,
            'most_received_resource': most_received
        })
        
    df_derived = pl.DataFrame(derived_metrics_list)
    
    # Join the derived columns back 
    player_profiles = player_profiles.join(df_derived, on='player', how='left')
    
    # Drop the intermediate aggregate columns to clean up the final profiled DataFrame
    columns_to_drop = [
        'total_starting_ore', 'total_starting_wool', 'total_starting_lumber', 
        'total_starting_grain', 'total_starting_brick',
        'total_traded_away_ore', 'total_traded_away_wool', 'total_traded_away_lumber', 
        'total_traded_away_grain', 'total_traded_away_brick',
        'total_received_trade_ore', 'total_received_trade_wool', 'total_received_trade_lumber', 
        'total_received_trade_grain', 'total_received_trade_brick'
    ]
    player_profiles = player_profiles.drop(columns_to_drop)
    
    # Sort by number of games played (descending)
    player_profiles = player_profiles.sort('num_games', descending=True)
    
    # Display the result
    print("PLAYER PROFILES:")
    display(player_profiles.head(9))
else:
    print("Could not generate player profiles because df_all is empty.")


PLAYER PROFILES:


player,num_games,avg_turns_before_first_trade,ratio_cards_given_to_taken,trade_success_rate,avg_counter_offers,avg_bank_trades,avg_dev_cards_bought,avg_roads_built,avg_cities_built,avg_players_targeted,avg_times_targeted,avg_cards_discarded_to_7,win_rate_largest_army,win_rate_longest_road,overall_avg_hand_size,top_3_starting_resources,most_traded_away_resource,most_received_resource
str,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str
"""HomeofAD3005""",11,20.181818,0.930233,0.207792,3.272727,5.727273,2.272727,7.636364,1.727273,2.181818,0.0,0.0,0.090909,0.545455,6.41842,"""lumber, grain, brick""","""wool""","""ore"""
"""Romeoore""",7,20.714286,1.026316,0.191781,3.0,3.571429,6.0,6.0,1.857143,2.285714,0.0,0.0,0.714286,0.142857,6.069925,"""grain, wool, ore""","""lumber""","""ore"""
"""UniQueLagacy""",5,18.6,1.04,0.174603,2.0,4.8,2.2,7.6,1.6,1.8,0.0,0.0,0.0,0.6,8.463008,"""grain, brick, wool""","""lumber""","""ore"""
"""AatNeverLose""",5,22.2,1.176471,0.2,0.0,3.6,4.0,8.0,1.8,1.8,0.0,0.0,0.2,0.0,7.253332,"""grain, lumber, brick""","""lumber""","""wool"""
"""GoldenGrain""",3,27.333333,0.866667,0.333333,1.0,6.333333,3.0,7.0,1.333333,1.333333,5.333333,0.0,0.0,0.0,7.058081,"""grain, lumber, wool""","""wool""","""grain"""
"""ZL24""",3,14.0,1.333333,0.5,1.333333,5.0,4.666667,5.0,2.333333,1.0,7.666667,0.0,0.0,0.0,7.29083,"""lumber, wool, grain""","""lumber""","""grain"""
"""GravityMarty""",2,18.0,0.857143,0.105263,1.0,3.0,4.5,4.0,1.5,1.0,6.0,0.0,0.0,0.0,4.764532,"""grain, wool, lumber""","""grain""","""grain"""
"""SirBeef""",2,13.0,0.625,0.230769,0.0,3.0,6.5,5.5,1.0,1.5,3.0,0.0,1.0,0.0,5.905195,"""wool, lumber, ore""","""wool""","""grain"""
"""Deleted User""",2,29.0,1.2,0.166667,0.5,3.0,4.0,6.0,2.0,1.5,4.5,0.0,0.0,0.5,5.620643,"""grain, ore, brick""","""wool""","""grain"""


In [22]:
player_profiles.columns


['player',
 'num_games',
 'avg_turns_before_first_trade',
 'ratio_cards_given_to_taken',
 'trade_success_rate',
 'avg_counter_offers',
 'avg_bank_trades',
 'avg_dev_cards_bought',
 'avg_roads_built',
 'avg_cities_built',
 'avg_players_targeted',
 'avg_times_targeted',
 'avg_cards_discarded_to_7',
 'win_rate_largest_army',
 'win_rate_longest_road',
 'overall_avg_hand_size',
 'top_3_starting_resources',
 'most_traded_away_resource',
 'most_received_resource']